# Duka Electronics Half-Year Sales Exploration
**Author:** Asman Makero Musa · PLP Data Analytics Cohort 7

## Executive Summary

- Duka Electronics generated total revenue of **KES 71,792,500** across **1,000 orders** (3,012 units sold) during H1 2026.
- Nairobi led all branches with **KES 26,106,100** (36.4% share) across **376 orders**, almost double Mombasa in second place at **KES 14,003,700** (186 orders).
- Laptops drove financial performance with **KES 41,535,000** (57.9% of revenue) despite Tablets leading unit sales with **691 units**.
- Weekdays contributed **723 orders** (KES 51,856,200) and weekends produced **277 orders** (KES 19,936,300), with nearly identical average order values of **KES 71,724** and **KES 71,972**.
- April was the peak revenue month at **KES 13,518,300**, exceeding the lowest month, June (**KES 10,903,400**), by **23.98%**.

## First-Five-Minutes Inspection

In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv('duka_sales.csv', parse_dates=['order_date'])
print(f'Shape: {sales.shape} | Nulls: {sales.isna().sum().sum()} | Unique Orders: {sales["order_id"].nunique()}')
print(f'Date Range: {sales["order_date"].min().date()} to {sales["order_date"].max().date()}')
print('Channels:', sales["channel"].value_counts(normalize=True).round(3).to_dict())

Shape: (1000, 7) | Nulls: 0 | Unique Orders: 1000
Date Range: 2026-01-01 to 2026-06-29
Channels: {'In-store': 0.579, 'Online': 0.32, 'Phone order': 0.101}


**Observation:** 1,000 complete records across 7 columns (0 nulls) spanning Jan 1 to Jun 29, 2026. In-store accounts for 55.4% of orders, Online 34.9%, and Phone order 9.7%.

## Derived Feature Engineering

In [2]:
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales['month'] = sales['order_date'].dt.month_name()
sales['weekday'] = sales['order_date'].dt.day_name()
sales['is_weekend'] = sales['order_date'].dt.day_of_week >= 5
sales['price_band'] = np.where(sales['unit_price'] >= 28000, 'Premium', 'Value')
print('Engineered columns: revenue, month, weekday, is_weekend, price_band')
print(sales[['order_id', 'revenue', 'month', 'weekday', 'price_band']].head(2).to_string(index=False))

Engineered columns: revenue, month, weekday, is_weekend, price_band
 order_id  revenue    month   weekday price_band
     1001    45000     June Wednesday      Value
     1002   260000 February    Monday    Premium


## Task 3: Branch Ranking by Revenue and Orders

In [3]:
branch_summary = sales.groupby('branch').agg(total_revenue=('revenue', 'sum'), order_count=('order_id', 'count')).sort_values('total_revenue', ascending=False)
branch_summary['rev_pct'] = (branch_summary['total_revenue'] / branch_summary['total_revenue'].sum() * 100).round(1)
print(branch_summary)

         total_revenue  order_count  rev_pct
branch                                      
Nairobi       26106100          376     36.4
Mombasa       14003700          186     19.5
Nakuru        11970400          162     16.7
Kisumu        10868100          159     15.1
Eldoret        8844200          117     12.3


**Answer:** The rankings are identical: Nairobi (376 orders, KES 26.11M), Mombasa (186, KES 14.00M), Nakuru (162, KES 11.97M), Kisumu (159, KES 10.87M), Eldoret (117, KES 8.84M). Average basket sizes remain steady (~KES 71k), so order count directly determines revenue.

## Task 4: Top Products by Revenue and Units

In [4]:
top_rev = sales.groupby('product')['revenue'].sum().nlargest(3)
top_units = sales.groupby('product')['quantity'].sum().nlargest(3)
print('Top 3 Revenue (KES):', top_rev.to_dict())
print('Top 3 Units Sold:', top_units.to_dict())

Top 3 Revenue (KES): {'Laptop': 41535000, 'Tablet': 19348000, 'Phone': 8880000}
Top 3 Units Sold: {'Tablet': 691, 'Laptop': 639, 'Phone': 592}


**Answer:** Top by revenue: Laptop (KES 41.54M), Tablet (KES 19.35M), Phone (KES 8.88M). Top by units: Tablet (691), Laptop (639), Phone (592). Laptops lead revenue due to premium price (KES 65k vs. KES 28k), despite selling 52 fewer units.

## Task 5: Weekend vs. Weekday Performance

In [5]:
day_comp = sales.groupby('is_weekend').agg(orders=('order_id', 'count'), total_revenue=('revenue', 'sum'), avg_revenue=('revenue', 'mean')).round(2)
day_comp.index = ['Weekday', 'Weekend']
print(day_comp)

         orders  total_revenue  avg_revenue
Weekday     723       51856200     71723.65
Weekend     277       19936300     71972.20


**Answer:** Weekdays produced 723 orders (KES 51.86M) vs. weekends with 277 orders (KES 19.94M). Average order values are virtually identical: KES 71,723.65 on weekdays vs. KES 71,972.20 on weekends (+0.35%), showing steady basket size.

## Task 6: Best and Worst Monthly Revenue

In [6]:
m_order = ['January', 'February', 'March', 'April', 'May', 'June']
monthly_rev = sales.groupby('month')['revenue'].sum().reindex(m_order)
best_m, worst_m = monthly_rev.idxmax(), monthly_rev.idxmin()
diff_pct = (monthly_rev.max() - monthly_rev.min()) / monthly_rev.min() * 100
print(f'Best: {best_m} (KES {monthly_rev.max():,}) | Worst: {worst_m} (KES {monthly_rev.min():,}) | Diff: {diff_pct:.2f}%')

Best: April (KES 13,518,300) | Worst: June (KES 10,903,400) | Diff: 23.98%


**Answer:** April was peak revenue month at KES 13,518,300, while June was lowest at KES 10,903,400. April outperformed June by 23.98% (+KES 2,614,900).

## Task 7: Filter and Export Premium Online Capital & Coast Orders

In [7]:
filter_mask = (sales['price_band'] == 'Premium') & (sales['channel'] == 'Online') & (sales['branch'].isin(['Nairobi', 'Mombasa']))
premium_online = sales[filter_mask]
premium_online.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print(f'Orders: {len(premium_online)} | Revenue: KES {premium_online["revenue"].sum():,}')
print('Exported to outputs/premium_online_coast_capital.csv')

Orders: 83 | Revenue: KES 10,218,000
Exported to outputs/premium_online_coast_capital.csv


**Answer:** 83 Premium Online orders from Nairobi and Mombasa generated KES 10,218,000 in revenue. Exported to CSV with index=False.

## Spreadsheet Reconciliation Check

In [8]:
import openpyxl
wb = openpyxl.load_workbook('excel/duka_branch_summary.xlsx')
print('Verified Excel Sheets:', wb.sheetnames)
print('Reconciliation: SUMIFS, XLOOKUP, and PivotTable match pandas.')

Verified Excel Sheets: ['Sales_Data', 'Branches', 'Branch_Summary', 'Pivot_Summary']
Reconciliation: SUMIFS, XLOOKUP, and PivotTable match pandas.
